# Stage 1 Data Preparation: COCO Vietnamese Captions + UIT-OpenViIC

This notebook prepares the captioning data used for Stage 1 projector alignment. It runs the project scripts and visualizes the generated JSONL files.

In [ ]:
!pip -q install -U gdown datasets pillow pandas pyarrow matplotlib seaborn tqdm pyyaml loguru beautifulsoup4 requests openai python-dotenv kaggle

import json
import os
import random
import shutil
import subprocess
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

In [ ]:
REPO_URL = "https://github.com/duongtruongbinh/pretrain_vlm.git"
BRANCH = "heisenburger"
PROJECT_DIR = Path("/content/pretrain_vlm")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)

os.chdir(PROJECT_DIR)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
print("Project root:", Path.cwd())

## Prepare Stage 1 datasets

`uit` downloads and prepares UIT-OpenViIC. `coco` streams `DavidPhilips/coco2017`, saves images locally, and writes Vietnamese caption JSONL files.

Set `RUN_FULL = False` for a quick Colab smoke run.

In [ ]:
RUN_FULL = True
COCO_SMOKE_ROWS = 200

subprocess.run(["python", "scripts/prepare_data.py", "uit"], check=True)

coco_cmd = ["python", "scripts/prepare_data.py", "coco"]
if not RUN_FULL:
    coco_cmd += ["--max-rows-per-split", str(COCO_SMOKE_ROWS), "--overwrite"]
subprocess.run(coco_cmd, check=True)

## Inspect prepared files

In [ ]:
def read_jsonl(path, limit=None):
    path = Path(path)
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
            if limit is not None and len(rows) >= limit:
                break
    return rows


def count_jsonl(path):
    path = Path(path)
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def plot_counts(title, counts):
    series = pd.Series(counts, dtype="int64")
    ax = series.plot(kind="bar", figsize=(7, 4), color="#4C78A8")
    ax.set_title(title)
    ax.set_ylabel("rows")
    ax.bar_label(ax.containers[0])
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def show_image_grid(records, text_getter, n=6):
    samples = [r for r in records if Path(str(r.get("image", ""))).exists()]
    if not samples:
        print("No local image samples found.")
        return
    samples = random.sample(samples, min(n, len(samples)))
    cols = min(3, len(samples))
    rows = (len(samples) + cols - 1) // cols
    plt.figure(figsize=(5 * cols, 4.5 * rows))
    for i, record in enumerate(samples, start=1):
        image = Image.open(record["image"]).convert("RGB")
        ax = plt.subplot(rows, cols, i)
        ax.imshow(image)
        ax.axis("off")
        ax.set_title(text_getter(record)[:120], fontsize=9)
    plt.tight_layout()
    plt.show()

stage1_paths = {
    "coco_train": "data/coco2017/train.jsonl",
    "coco_val": "data/coco2017/val.jsonl",
    "uit_train": "data/uit-openviic/train.jsonl",
    "uit_val": "data/uit-openviic/val.jsonl",
    "uit_test": "data/uit-openviic/test.jsonl",
}
counts = {name: count_jsonl(path) for name, path in stage1_paths.items()}
display(pd.DataFrame([{"split": k, "rows": v} for k, v in counts.items()]))
plot_counts("Stage 1 prepared caption rows", counts)

## Visualize random caption samples

In [ ]:
sample_rows = []
for path in stage1_paths.values():
    sample_rows.extend(read_jsonl(path, limit=200))

show_image_grid(sample_rows, lambda r: r.get("caption", ""), n=6)